# Agent 365 Registry — Ingester (Fabric) · ⚠️ PREVIEW / OPTIONAL

> **Status: PREVIEW — unvalidated against a live tenant.** This notebook calls the **beta** Microsoft
> Graph *Agent 365 catalog* endpoint that Microsoft's own PAX tool shipped in v1.11.1 and then
> **retired in v1.11.2** because the upstream surface (server-side paging, role gating, app-only
> support) had not stabilised. Treat it as a starting point, not a finished pipeline. For a
> dependable path today, use **`Copilot_Agent365_Lander.ipynb`** (lands the Agent 365 admin-center
> **export CSV**) — that is the supported source and is what PAX reverted to.

## What this would do

Pulls the tenant's **Agent 365 agent catalog** (declarative agents, plugins, etc. — including
their data-access **capabilities/permissions**: OneDrive/SharePoint read, Graph connector, code
interpreter, image generation, uploaded files) into a Lakehouse Delta table `dbo.agent365_catalog`,
keyed on `Title ID` (the `T_`-prefixed titleId). The dashboard's **Agents 365** page can then read
the richer capability columns the plain export may not carry.

## Why it's optional (read before using)

| Constraint | Impact |
|---|---|
| **Endpoint is `beta` only** (`/copilot/admin/catalog/packages`) | May change/break without notice — no `/v1.0` as of 2026. |
| **Delegated auth only — no app-only / service principal** | **Cannot run unattended / on a schedule.** A user with the right role must sign in interactively each run. This is the big one for an automated dashboard. |
| **Requires Agent 365 *Frontier program* enrolment** + **AI Administrator** (or Global Admin) role | Niche; most tenants get 401/403/404. The user's note that "A365 licences are required" is consistent with this. |
| **Point-in-time only** | No history; deleted agents disappear; `Date created`/`Created by` need a separate audit-log join. |

**Recommendation:** keep the **export lander** as the default Agent 365 path; use this notebook only
in a Frontier tenant where you specifically want the live capability/permission detail, and run it
**by hand** (it can't be a scheduled, service-principal job).

## Permissions / setup

- An **app registration** consented for delegated scopes **`CopilotPackages.Read.All`** and
  **`Application.Read.All`**, and the signed-in user holding **AI Administrator** (or Global Admin).
- Install MSAL in the Spark session if not present: `%pip install msal`.

*Source: Microsoft PAX v1.11.1 (`MICROSOFT AGENT 365 ENRICHMENT FUNCTIONS` block) — endpoint, 28-column
schema and auth recovered from the released script. Ref: `learn.microsoft.com/en-us/microsoft-agent-365/admin/graph-api` (limited early access).*

## 1. Configuration & delegated sign-in

Delegated **device-code** flow (works in a notebook — no browser redirect needed). App-only is
**not** supported by the endpoint, so there is no client-secret path here.

In [ ]:
# === CONFIG ===
TENANT_ID = '<your-tenant-guid>'
CLIENT_ID = '<app-reg-client-id>'        # consented for CopilotPackages.Read.All + Application.Read.All
TARGET_TABLE = 'dbo.agent365_catalog'
WRITE_MODE   = 'overwrite'

# Delegated device-code sign-in (interactive — paste the code at https://microsoft.com/devicelogin)
# %pip install msal   # uncomment on first run if msal isn't available
import msal
_app = msal.PublicClientApplication(CLIENT_ID, authority=f'https://login.microsoftonline.com/{TENANT_ID}')
_scopes = ['CopilotPackages.Read.All', 'Application.Read.All']
_flow = _app.initiate_device_flow(scopes=_scopes)
print(_flow['message'])          # -> 'To sign in, use a web browser to open ... and enter the code ...'
_result = _app.acquire_token_by_device_flow(_flow)   # blocks until you complete sign-in
TOKEN = _result['access_token']
print('signed in:', 'access_token' in _result)

## 2. Call the beta catalog endpoint

List → page via `@odata.nextLink` → fetch per-package detail (the `elementDetails` capability fields
only appear at the detail level). A 401/403/404 here means Frontier enrolment or the AI Administrator
role is missing — not a code bug.

In [ ]:
import requests

BASE = 'https://graph.microsoft.com/beta/copilot/admin/catalog/packages'
H = {'Authorization': f'Bearer {TOKEN}'}

# Probe (clear message if not enrolled / wrong role)
probe = requests.get(f'{BASE}?$top=1', headers=H)
if probe.status_code in (401, 403, 404):
    raise PermissionError(f'{probe.status_code}: Agent 365 catalog not accessible. Requires Frontier '
                          'enrolment + AI Administrator role + CopilotPackages.Read.All consent.')

# Page the catalog list
packages, url = [], BASE
for _ in range(500):                       # safety cap (paging was flaky in PAX)
    r = requests.get(url, headers=H); r.raise_for_status()
    data = r.json(); packages.extend(data.get('value', []))
    url = data.get('@odata.nextLink')
    if not url:
        break
print('packages in catalog:', len(packages))

## 3. Map to the 28-column schema → Delta

Column names match the PAX `ConvertTo-Agent365Row` output so the dashboard's `Agents 365` query can
read them. `Title ID` is the primary/merge key. `Date created` / `Created by` are left blank here
(they need a separate Purview audit-log join — out of scope for this preview).

In [ ]:
from pyspark.sql import functions as F

def _join(v):
    return ';'.join(v) if isinstance(v, list) else (v if v is not None else '')

rows = []
for pkg in packages:
    pid = pkg.get('id') or pkg.get('titleId') or pkg.get('packageId')
    d = requests.get(f'{BASE}/{pid}', headers=H)
    detail = d.json() if d.ok else pkg
    elem = detail.get('elementDetails') or {}
    tid = detail.get('id') or detail.get('titleId') or detail.get('packageId') or ''
    title_id = tid if str(tid).startswith('T_') else f'T_{tid}'
    rows.append({
        'Name': detail.get('displayName') or detail.get('name', ''),
        'Supported in': _join(detail.get('supportedHosts') or detail.get('supportedClients') or []),
        'Date created': '',
        'Developer Name': (detail.get('developer') or {}).get('name', ''),
        'Type': detail.get('agentType') or detail.get('type', ''),
        'Version': detail.get('version', ''),
        'Availability': _join(detail.get('availability') or detail.get('allowedUsersAndGroups') or ''),
        'Created by': '',
        'Description': detail.get('description', ''),
        'Created in': detail.get('source') or detail.get('origin') or detail.get('createdIn', ''),
        'Last updated': detail.get('lastModifiedDateTime') or detail.get('lastUpdatedDateTime', ''),
        'Custom actions': _join(elem.get('customActions', '')),
        'Title ID': title_id,
        'Sensitivity': detail.get('sensitivity', ''),
        'Can read OneDrive and Sharepoint items': elem.get('canReadOneDriveAndSharepointItems', ''),
        'OneDrive and Sharepoint items': _join(elem.get('oneDriveAndSharepointItems', '')),
        'Can read OneDrive files': elem.get('canReadOneDriveFiles', ''),
        'OneDrive files': _join(elem.get('oneDriveFiles', '')),
        'OneDrive sites': _join(elem.get('oneDriveSites', '')),
        'Can read Sharepoint sites and files': elem.get('canReadSharepointSitesAndFiles', ''),
        'Sharepoint files': _join(elem.get('sharepointFiles', '')),
        'Sharepoint sites': _join(elem.get('sharepointSites', '')),
        'Can extend to Graph connector': elem.get('canExtendToGraphConnector', ''),
        'Graph connector details': _join(elem.get('graphConnectorDetails', '')),
        'Can generate images using user prompt': elem.get('canGenerateImagesUsingUserPrompt', ''),
        'Can use code interpreter': elem.get('canUseCodeInterpreter', ''),
        'Contains uploaded files': elem.get('containsUploadedFiles', ''),
        'Uploaded files': _join(elem.get('uploadedFiles', '')),
    })

# everything as string for a clean, schema-stable Delta write
rows = [{k: ('' if v is None else str(v)) for k, v in r.items()} for r in rows]
df = spark.createDataFrame(rows) if rows else spark.createDataFrame([], 'Name string')

(df.write.mode(WRITE_MODE)
   .option('overwriteSchema', 'true')
   .option('delta.columnMapping.mode', 'name')      # spaced column names
   .option('delta.minReaderVersion', '2')
   .option('delta.minWriterVersion', '5')
   .format('delta').saveAsTable(TARGET_TABLE))

print(f'{TARGET_TABLE}: {df.count():,} agents written.')
print('NOTE: PREVIEW — verify column names against the Agents 365 model query before relying on it.')